# 03 — Fine-tuning LoRA/QLoRA de Gemma E2B (Phase 3)

Correspond à la Phase 3 de `docs/WORKFLOW.md`. Charge les fichiers produits par `02_build_dataset.ipynb` (`train_balanced.jsonl`, `dev_split.jsonl` sur Drive) et fine-tune `google/gemma-4-E2B-it` en QLoRA sur T4.

**Résultats de la Phase 2 réutilisés ici** : dataset rééquilibré à 53 300 lignes, `MAX_SEQ_LENGTH` p95 mesuré à 477 tokens → on utilise **512** avec un peu de marge.

## 1 — Environnement

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/gemmafro-e2b'
PROCESSED_DIR = f'{PROJECT_DIR}/processed_data'
assert os.path.isdir(PROCESSED_DIR), "processed_data introuvable — exécuter 02_build_dataset.ipynb d'abord."

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

## 2 — Charger le dataset traité (Phase 2)

In [ ]:
from datasets import load_dataset

# Chargés séparément (pas via un même dict data_files) : train_balanced.jsonl et
# dev_split.jsonl n'ont pas exactement les mêmes colonnes, et datasets tente sinon
# d'unifier le schéma entre tous les fichiers d'un même appel, ce qui casse le chargement.
train_ds = load_dataset('json', data_files=f'{PROCESSED_DIR}/train_balanced.jsonl', split='train')
dev_ds   = load_dataset('json', data_files=f'{PROCESSED_DIR}/dev_split.jsonl', split='train')
print(train_ds)
print(dev_ds)

## 3 — Charger le tokenizer et le modèle en 4-bit (QLoRA)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "google/gemma-4-E2B-it"
MAX_SEQ_LENGTH = 512  # p95 mesuré en Phase 2 = 477 ; marge incluse

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# Padding à droite : le masquage de la perte ci-dessous indexe les tokens depuis le début de séquence.
tokenizer.padding_side = 'right'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.pad_token_id = tokenizer.pad_token_id

## 4 — Configurer LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5 — Collator avec masquage de perte par longueur mesurée

`full_text` = prompt + réponse ; `prompt_text` = prompt seul (mêmes textes que ceux produits en Phase 2). On tokenise les deux et on masque (`label = -100`) les `len(prompt_text)` premiers tokens de `full_text`, pour que la perte ne porte que sur la réponse générée — sans dépendre du texte exact des tokens spéciaux du template (qui s'est avéré être `<|turn>...<turn|>` pour ce checkpoint, différent du `<start_of_turn>` d'autres versions de Gemma).

In [ ]:
import torch

class PromptMaskingCollator:
    def __init__(self, tokenizer, max_length):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, batch):
        full_texts = [ex['full_text'] for ex in batch]
        prompt_texts = [ex['prompt_text'] for ex in batch]

        enc = self.tokenizer(
            full_texts,
            add_special_tokens=False,  # les tokens spéciaux sont déjà écrits dans le texte du template
            truncation=True,
            max_length=self.max_length,
            padding=True,
            return_tensors='pt',
        )
        labels = enc['input_ids'].clone()

        for i, prompt in enumerate(prompt_texts):
            prompt_len = len(self.tokenizer(prompt, add_special_tokens=False)['input_ids'])
            prompt_len = min(prompt_len, labels.shape[1])
            labels[i, :prompt_len] = -100
        labels[enc['attention_mask'] == 0] = -100

        return {
            'input_ids': enc['input_ids'],
            'attention_mask': enc['attention_mask'],
            'labels': labels,
        }

collator = PromptMaskingCollator(tokenizer, MAX_SEQ_LENGTH)

## 6 — Configuration et lancement de l'entraînement

`per_device_train_batch_size=2` + `gradient_accumulation_steps=8` (batch effectif 16) : point de départ prudent pour un T4 16 Go avec le modèle déjà à ~6,7 Go en 4-bit. Réduire à 1 si OOM, augmenter si de la VRAM reste libre (`!nvidia-smi`).

In [ ]:
from transformers import TrainingArguments, Trainer
from transformers.trainer_utils import get_last_checkpoint

OUTPUT_DIR = f'{PROJECT_DIR}/checkpoints/gemma-4-e2b-lora'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,  # pas bf16 : non supporté efficacement sur T4
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,  # limite l'espace Drive utilisé par les checkpoints intermédiaires
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=collator,
)

In [ ]:
# Reprise automatique si une session précédente a été interrompue (déconnexion Colab) —
# cf. contraintes du GPU gratuit en Phase 0 de docs/WORKFLOW.md.
last_checkpoint = get_last_checkpoint(OUTPUT_DIR) if os.path.isdir(OUTPUT_DIR) else None
if last_checkpoint:
    print(f'Reprise depuis {last_checkpoint}')
else:
    print('Nouvel entraînement (aucun checkpoint existant)')

trainer.train(resume_from_checkpoint=last_checkpoint)

## 7 — Courbe de perte

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'axes.edgecolor': '#c3c2b7',
    'axes.labelcolor': '#52514e',
    'text.color': '#0b0b0b',
    'xtick.color': '#898781',
    'ytick.color': '#898781',
    'axes.grid': True,
    'grid.color': '#e1e0d9',
    'grid.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.axisbelow': True,
})

history = pd.DataFrame(trainer.state.log_history)
train_hist = history.dropna(subset=['loss'])[['step', 'loss']] if 'loss' in history else pd.DataFrame()
eval_hist = history.dropna(subset=['eval_loss'])[['step', 'eval_loss']] if 'eval_loss' in history else pd.DataFrame()

fig, ax = plt.subplots(figsize=(9, 5))
if not train_hist.empty:
    ax.plot(train_hist['step'], train_hist['loss'], color='#2a78d6', linewidth=2, label='Train loss')
if not eval_hist.empty:
    ax.plot(eval_hist['step'], eval_hist['eval_loss'], color='#eb6834', linewidth=2, label='Eval loss')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Courbe de perte — fine-tuning Gemma E2B')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## 8 — Sauvegarder l'adaptateur LoRA final

In [ ]:
FINAL_DIR = f'{PROJECT_DIR}/checkpoints/gemma-4-e2b-lora-final'
trainer.model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print('Adaptateur LoRA final sauvegardé dans', FINAL_DIR)

---
**Étape suivante : Phase 4 — Évaluation** (ROUGE-1/ROUGE-L sur `Val.csv` par langue, comparaison au zero-shot) à partir de l'adaptateur sauvegardé dans `FINAL_DIR`.